In [0]:
from pyspark.sql.functions import (col,upper,trim,concat_ws,coalesce,lit,udf,when,max,create_map,to_timestamp,date_sub,to_date,first,lower,size,split,initcap,element_at)
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number
from pyspark.sql import Row
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
tables=spark.catalog.listTables("f1_warehouse.gold")

dim={}

for table in tables:
    df=spark.table(f"f1_warehouse.gold.{table.name}")
    dim[table.name]=df

In [0]:
for table_names,df in dim.items():
    print(table_names)

In [0]:
tables=spark.catalog.listTables("f1_warehouse.silver")

silver={}

for table in tables:
    df=spark.table(f"f1_warehouse.silver.{table.name}")
    silver[table.name]=df

In [0]:
for table_names,df in silver.items():
    print(table_names)

### fact_results

In [0]:
of1_race_results = silver["openf1_race_results"]

In [0]:
 
# race_meeting_id
of1_race_results = (
    of1_race_results.join(
        dim["dim_meetings"].select("openf1_meeting_id", "race_meeting_id","date_start"),
        of1_race_results["meeting_id"] == dim["dim_meetings"]["openf1_meeting_id"],
        "left"
    )
    .drop("openf1_meeting_id")
)

In [0]:
#constructor_id
of1_drivers=silver["openf1_drivers"].alias("of1")
teams = dim["dim_teams"].select(
    "name",
    "constructor_id"
).alias("t")

of1_drivers = (
    of1_drivers
    .join(
        teams,
        col("of1.team_name") == col("t.name"),
        how="left"
    )
    .drop(col("t.name"))
)

In [0]:
of1_drivers=of1_drivers.select("driver_number","first_name","full_name","last_name","session_id","meeting_id","constructor_id")

In [0]:
#join of1 drivers and dim drivers on name acronym
# driver id

of1_cleaned_driver_names = (
    of1_drivers
    .withColumn(
        "first_name",
        when(
            col("first_name").isNull(),
            initcap(trim(element_at(split(col("full_name"), " "), 1)))
        ).otherwise(col("first_name"))
    )
    .withColumn(
        "last_name",
        when(
            col("last_name").isNull(),
            initcap(trim(element_at(split(col("full_name"), " "), -1)))
        ).otherwise(col("last_name"))
    )
)

In [0]:

#join driver id on first name,last name and number

drivers = dim["dim_driver"].withColumn("number", col("number").try_cast("int"))

of1_cleaned_drivers = (
    of1_cleaned_driver_names.alias("of1")
    .join(
        drivers.alias("d"),
        (col("of1.first_name") == col("d.forename")) &
        (col("of1.last_name") == col("d.surname")) &
        (col("of1.driver_number") == col("d.number")),
        "inner"
    )
    .drop("d.number", "d.surname","d.forename")
)

In [0]:
of1_race_results = (
    of1_race_results.alias("r")
    .join(
        of1_cleaned_drivers.select(
            "session_id",
            "driver_number",
            "constructor_id",
            "driver_id"
        ).alias("d"),
        (col("r.session_id") == col("d.session_id")) &
        (col("r.driver_number") == col("d.driver_number")),
        how="left"
    )
    .drop(col("d.session_id"))
    .drop(col("d.driver_number"))
)

In [0]:
#join starting grid to openf1

of1_race_results=(
    of1_race_results
    .join(
        silver["openf1_starting_grid"]
            .select(
                "meeting_id",
                "driver_number",
                col("position").alias("starting_position")
            ),
        on=["meeting_id", "driver_number"],
        how="left"
    )
)

In [0]:
incoming_race_results = of1_race_results.dropDuplicates()

In [0]:
existing_race_results=dim["fact_race_results"]

new_race_results = (
    incoming_race_results.alias("n")
    .join(
        existing_race_results.alias("f"),
        (F.col("n.race_meeting_id") == F.col("f.race_meeting_id")) &
        (F.col("n.driver_id") == F.col("f.driver_id")) &
        (F.col("n.constructor_id") == F.col("f.constructor_id")),
        "leftanti"
    )
)


In [0]:
new_race_results.display()

In [0]:
new_race_results.printSchema()

In [0]:
new_race_results=new_race_results.withColumn("nc",col("position").isNull() & ~(col("dnf") | col("dns") | col("dsq")))

In [0]:
existing_race_results.printSchema()


In [0]:
new_race_results = new_race_results.select("race_meeting_id","driver_id","constructor_id","starting_position","position","gap_to_leader","duration","points","number_of_laps","dnf","dns","dsq","nc")

In [0]:
#merge into gold fact

delta_fact_race = DeltaTable.forName(
    spark,
    "f1_warehouse.gold.fact_race_results"
)

(
    delta_fact_race.alias("f")
    .merge(
        new_race_results.alias("n"),
        """
        f.race_meeting_id = n.race_meeting_id
        AND f.driver_id = n.driver_id
        AND f.constructor_id = n.constructor_id
        """
    )
    .whenNotMatchedInsertAll()
    .execute()
)

In [0]:
delta_fact_race.toDF().count()

### fact_pit

In [0]:
of1_pit = silver["openf1_pit"]
of1_drivers=silver["openf1_drivers"]


In [0]:
of1_drivers=of1_drivers.select("driver_number","first_name","full_name","last_name","session_id","meeting_id")

In [0]:
#join of1 drivers and dim drivers on name acronym
# driver id

of1_cleaned_driver_names = (
    of1_drivers
    .withColumn(
        "first_name",
        when(
            col("first_name").isNull(),
            initcap(trim(element_at(split(col("full_name"), " "), 1)))
        ).otherwise(col("first_name"))
    )
    .withColumn(
        "last_name",
        when(
            col("last_name").isNull(),
            initcap(trim(element_at(split(col("full_name"), " "), -1)))
        ).otherwise(col("last_name"))
    )
)

In [0]:

#join driver id on first name,last name and number

drivers = dim["dim_driver"].withColumn("number", col("number").try_cast("int"))

of1_cleaned_drivers = (
    of1_cleaned_driver_names.alias("of1")
    .join(
        drivers.alias("d"),
        (col("of1.first_name") == col("d.forename")) &
        (col("of1.last_name") == col("d.surname")) &
        (col("of1.driver_number") == col("d.number")),
        "inner"
    )
    .drop("d.number", "d.surname","d.forename")

)

In [0]:
#join meeting key

of1_pit= (
    of1_pit.join(
        dim["dim_meetings"].select("openf1_meeting_id", "race_meeting_id"),
        of1_pit["meeting_id"] == dim["dim_meetings"]["openf1_meeting_id"],
        "left"
    )
    .drop("openf1_meeting_id")
)

In [0]:
#drop test drivers/reserved drivers
#join name acronym on driver number and session id
of1_pit = (
    of1_pit.alias("p")
    .join(
        of1_cleaned_drivers.select(
            "driver_number",
            "session_id",
            "driver_id",
        ).alias("d"),
        (col("p.session_id") == col("d.session_id")) &
        (col("p.driver_number") == col("d.driver_number")),
        how="inner"
    )
    .drop(col("d.session_id"))
    .drop(col("d.driver_number"))
)

In [0]:
incoming_pit_stops=of1_pit.dropDuplicates()

existing_pit=dim["fact_pit_stops"]

new_pit_stops = (
    incoming_pit_stops.alias("n")
    .join(
        existing_pit.alias("f"),
        (F.col("n.race_meeting_id") == F.col("f.race_meeting_id")) &
        (F.col("n.driver_id") == F.col("f.driver_id")),
        "leftanti"
    )
)


In [0]:
new_pit_stops.count()

In [0]:
new_pit_stops.show()

In [0]:
new_pit_stops = new_pit_stops.select(
    "driver_id",
    "race_meeting_id",
    lit(None).cast("int").alias("stop_number"),
    "lap_number",
    col("pit_duration").cast("double").alias("stop_duration"),
    col("date").alias("stop_time")
)

In [0]:
new_pit_stops.printSchema()

In [0]:
existing_pit.count()

In [0]:
existing_pit.printSchema()

In [0]:
delta_fact_pit = DeltaTable.forName(
    spark,
    "f1_warehouse.gold.fact_pit_stops"
)

(
    delta_fact_pit.alias("f")
    .merge(
        new_pit_stops.alias("n"),
        """
        f.race_meeting_id = n.race_meeting_id
        AND f.driver_id = n.driver_id
        """
    )
    .whenNotMatchedInsertAll()
    .execute()
)

In [0]:
delta_fact_pit.toDF().count()

### fact_qualifying

In [0]:
of1_qualifying=silver["openf1_qualifying_results"]

In [0]:
# join on name
#driver dim does not have constuctor id as the team u belong to can continuously change 
of1_drivers=silver["openf1_drivers"].alias("of1")
teams = dim["dim_teams"].select(
    "name",
    "constructor_id"
).alias("t")

of1_drivers = (
    of1_drivers
    .join(
        teams,
        col("of1.team_name") == col("t.name"),
        how="left"
    )
    .drop(col("t.name"))
)

In [0]:
of1_drivers=of1_drivers.select("driver_number","first_name","full_name","last_name","session_id","meeting_id","constructor_id")

In [0]:
#join of1 drivers and dim drivers on name acronym
# driver id

of1_cleaned_driver_names = (
    of1_drivers
    .withColumn(
        "first_name",
        when(
            col("first_name").isNull(),
            initcap(trim(element_at(split(col("full_name"), " "), 1)))
        ).otherwise(col("first_name"))
    )
    .withColumn(
        "last_name",
        when(
            col("last_name").isNull(),
            initcap(trim(element_at(split(col("full_name"), " "), -1)))
        ).otherwise(col("last_name"))
    )
)

In [0]:
#join driver id on first name,last name and number

drivers = dim["dim_driver"].withColumn("number", col("number").try_cast("int"))

of1_cleaned_drivers = (
    of1_cleaned_driver_names.alias("of1")
    .join(
        drivers.alias("d"),
        (col("of1.first_name") == col("d.forename")) &
        (col("of1.last_name") == col("d.surname")) &
        (col("of1.driver_number") == col("d.number")),
        "inner"
    )
    .drop("d.number", "d.surname","d.forename")

)

In [0]:
#join constructor id and driver id for openf1

#join driver key 

of1 = of1_qualifying.alias("of1")
of1_cleaned_drivers = of1_cleaned_drivers.select("driver_id", "driver_number", "constructor_id","session_id").alias("d")

of1_qualifying = (
    of1.join(of1_cleaned_drivers,
        (col("of1.session_id") == col("d.session_id")) &
        (col("of1.driver_number") == col("d.driver_number")),
        how="left"
    )
    .drop(col("d.driver_number"))   # only drops the dimension copy
)


In [0]:
#add meeting_key
of1_qualifying = (
    of1_qualifying.join(
        dim["dim_meetings"].select("openf1_meeting_id", "race_meeting_id"),
        of1_qualifying["meeting_id"] == dim["dim_meetings"]["openf1_meeting_id"],
        "left"
    )
    .drop("openf1_meeting_id")
)

In [0]:
#change string array to spark array
from pyspark.sql.functions import split, regexp_replace, trim, col

of1_qualifying = (
    of1_qualifying
    .withColumn("duration_array",split(regexp_replace(regexp_replace(col("duration"), r"[\[\]]", ""), " ", ""),",")
    )
)

In [0]:
from pyspark.sql.functions import try_element_at, lit

def time_to_seconds(column):
    parts = split(col(column), ":")
    return (
        try_element_at(parts, lit(1)).try_cast("double") * 60
        + try_element_at(parts, lit(2)).try_cast("double")
    )

In [0]:
from pyspark.sql.functions import expr
#split duration into q1,q2,q3
of1_qualifying = of1_qualifying.select(
    "driver_id",
    "race_meeting_id",
    "constructor_id",
    "position",
    expr("try_element_at(duration_array, 1)").try_cast("double").alias("q1"),
    expr("try_element_at(duration_array, 2)").try_cast("double").alias("q2"),
    expr("try_element_at(duration_array, 3)").try_cast("double").alias("q3"),
    "number_of_laps",
)

In [0]:
incoming_qualifying=of1_qualifying.dropDuplicates()

In [0]:
existing_qualifying=dim["fact_qualifying"]

new_qualifying = (
    incoming_qualifying.alias("n")
    .join(
        existing_qualifying.alias("f"),
        (F.col("n.race_meeting_id") == F.col("f.race_meeting_id")) &
        (F.col("n.driver_id") == F.col("f.driver_id")),
        "leftanti"
    )
)

In [0]:
new_qualifying.count()

In [0]:
existing_qualifying.count()

In [0]:
delta_fact_qualifying = DeltaTable.forName(
    spark,
    "f1_warehouse.gold.fact_qualifying"
)

(
    delta_fact_qualifying.alias("f")
    .merge(
        new_qualifying.alias("n"),
        """
        f.race_meeting_id = n.race_meeting_id
        AND f.driver_id = n.driver_id
        """
    )
    .whenNotMatchedInsertAll()
    .execute()
)

In [0]:
delta_fact_qualifying.toDF().count()